# 02A - Receivables / Invoice-Finance LGD

This notebook adds a dedicated receivables/invoice-finance segment under the parent commercial cash-flow framework.

Segment definition:
- Lending supported by accounts receivable / invoice pools where borrowing capacity depends on eligible receivables and advance rates.


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_commercial_data
from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(52)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 140)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
TABLE_DIR = os.path.join(OUTPUT_DIR, 'tables')
FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")


## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy:** observed workout inputs -> approved proxy inputs -> conservative fallback with disclosure.
- **Proxy logic:** receivables pool quality and collections control are represented with transparent synthetic proxies.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status:** this is a portfolio-project proxy baseline and is not calibrated to internal workout tapes.


## Objective
Build an interview-ready receivables / invoice-finance LGD segment with explicit EAD-headroom behaviour and receivables-quality drivers.

## Drivers
- Gross receivables, eligible receivables, ineligible receivables %
- Debtor concentration
- Ageing deterioration
- Dilution / credit-note proxy
- Collections control proxy (lockbox/sweep strength)
- Advance rate and time to collect

## Logic
- EAD is driven by drawings against eligible receivables and headroom utilisation near default.
- LGD depends on concentration, ageing, dilution, control weakness, and collection timing.

## Downturn
- Lower eligible pool %, slower collections, higher disputes/dilution, and stressed drawdown behaviour.

## Validation
- Balance consistency checks, EAD floor checks, and recovery timing sanity checks.

## Limitations
- Synthetic proxy data and expert-judgement overlays; internal workout calibration remains future work.


## Why This Segment Differs from Generic Term Lending

- Collateral is **short-cycle current assets** (receivables), not mainly hard assets.
- **Dilution and ageing** directly reduce collectible value.
- **Collections control** (lockbox/sweep discipline) affects leakage risk.
- EAD depends on **eligible receivables and advance rates**, not only facility limit and term amortisation.


In [ ]:
# Base commercial portfolio and receivables segment selection
all_loans, _ = generate_commercial_data(n_loans=420, seed=43)
all_loans = all_loans.copy()

all_loans['icr'] = pd.to_numeric(all_loans['interest_coverage_ratio'], errors='coerce')
all_loans['industry_risk_bucket'] = all_loans['industry_risk_level'].fillna('Medium')
all_loans['watchlist_flag'] = (
    all_loans['default_trigger'].isin(['Covenant Breach', 'Voluntary Administration', 'Receivership'])
    | (all_loans['icr'] < 1.35)
).astype(int)

# Receivables / invoice-finance proxy definition:
# - direct receivables security
# - mixed security where revolving behaviour suggests receivables-led draw
all_loans['receivables_dominant_proxy'] = (
    all_loans['security_type'].eq('PPSR - Receivables')
    | (
        all_loans['security_type'].eq('PPSR - Mixed')
        & all_loans['facility_type'].isin(['Revolving Credit', 'Overdraft'])
    )
).astype(int)

recv = all_loans[all_loans['receivables_dominant_proxy'] == 1].copy()

print('All loans:', len(all_loans))
print('Receivables segment loans:', len(recv))
display(recv[['loan_id', 'facility_type', 'security_type', 'ead']].head(8))



In [ ]:
# Build receivables-specific pool quality proxies
rng = np.random.default_rng(52)

recv['gross_receivables_balance'] = recv['ead'] * rng.uniform(1.10, 1.90, len(recv))

industry_risk_factor = recv['industry_risk_bucket'].map({'Low': 0.0, 'Medium': 0.5, 'Elevated': 1.0}).fillna(0.5)

recv['ineligible_receivables_pct'] = (
    0.08
    + 0.10 * recv['watchlist_flag']
    + 0.06 * industry_risk_factor
    + rng.normal(0.0, 0.03, len(recv))
).clip(0.03, 0.45)

recv['eligible_receivables_balance'] = (
    recv['gross_receivables_balance'] * (1.0 - recv['ineligible_receivables_pct'])
).clip(lower=0.0)

recv['debtor_concentration_pct'] = (
    0.15
    + 0.35 * industry_risk_factor
    + 0.18 * recv['watchlist_flag']
    + rng.normal(0.0, 0.08, len(recv))
).clip(0.05, 0.95)

recv['ageing_over_90d_pct'] = (
    0.06
    + 0.12 * recv['watchlist_flag']
    + 0.08 * industry_risk_factor
    + rng.normal(0.0, 0.04, len(recv))
).clip(0.01, 0.50)

recv['dilution_pct'] = (
    0.03
    + 0.08 * recv['watchlist_flag']
    + 0.06 * industry_risk_factor
    + rng.normal(0.0, 0.025, len(recv))
).clip(0.005, 0.30)

recv['collections_control_score'] = (
    0.78
    - 0.20 * recv['watchlist_flag']
    - 0.12 * industry_risk_factor
    + rng.normal(0.0, 0.08, len(recv))
).clip(0.05, 1.00)

recv['lockbox_sweep_flag'] = (recv['collections_control_score'] >= 0.62).astype(int)

recv['advance_rate'] = (
    0.82
    - 0.10 * recv['ineligible_receivables_pct']
    - 0.06 * recv['debtor_concentration_pct']
    - 0.05 * recv['dilution_pct']
    + 0.03 * recv['lockbox_sweep_flag']
    + rng.normal(0.0, 0.025, len(recv))
).clip(0.45, 0.90)

recv['time_to_collect_days'] = (
    42
    + 90 * recv['ageing_over_90d_pct']
    + 60 * recv['dilution_pct']
    + 25 * (1.0 - recv['collections_control_score'])
    + rng.normal(0.0, 10.0, len(recv))
).clip(20.0, 220.0)

# Banding for reporting
recv['concentration_band'] = pd.cut(
    recv['debtor_concentration_pct'],
    bins=[0.0, 0.25, 0.40, 0.60, 1.0],
    labels=['<=25%', '25-40%', '40-60%', '60%+'],
    right=True,
)
recv['ageing_band'] = pd.cut(
    recv['ageing_over_90d_pct'],
    bins=[0.0, 0.08, 0.15, 0.25, 1.0],
    labels=['<=8%', '8-15%', '15-25%', '25%+'],
    right=True,
)
recv['advance_rate_band'] = pd.cut(
    recv['advance_rate'],
    bins=[0.0, 0.60, 0.70, 0.80, 1.0],
    labels=['<=60%', '60-70%', '70-80%', '80%+'],
    right=True,
)

display(
    recv[
        [
            'gross_receivables_balance',
            'eligible_receivables_balance',
            'ineligible_receivables_pct',
            'debtor_concentration_pct',
            'ageing_over_90d_pct',
            'dilution_pct',
            'collections_control_score',
            'advance_rate',
            'time_to_collect_days',
        ]
    ].describe().round(4)
)


In [ ]:
# EAD construction: drawn + expected headroom usage against eligible receivables
recv['borrowing_base'] = recv['eligible_receivables_balance'] * recv['advance_rate']
recv['headroom_available'] = (recv['borrowing_base'] - recv['drawn_balance']).clip(lower=0.0)

recv['headroom_utilisation_near_default'] = (
    0.35
    + 0.30 * recv['watchlist_flag']
    + 0.18 * (1.0 - recv['collections_control_score'])
    + 0.15 * recv['debtor_concentration_pct']
).clip(0.10, 1.00)

recv['ead_base'] = recv['drawn_balance'] + recv['headroom_available'] * recv['headroom_utilisation_near_default']
recv['ead_base'] = np.maximum(recv['ead_base'], recv['drawn_balance'])

# Downturn EAD: weaker eligibility and higher drawdown pressure
recv['ineligible_receivables_pct_downturn'] = (
    recv['ineligible_receivables_pct']
    + 0.05
    + 0.06 * recv['ageing_over_90d_pct']
    + 0.04 * recv['dilution_pct']
).clip(0.05, 0.70)
recv['eligible_receivables_balance_downturn'] = recv['gross_receivables_balance'] * (1.0 - recv['ineligible_receivables_pct_downturn'])

recv['advance_rate_downturn'] = (
    recv['advance_rate']
    - 0.04
    - 0.03 * recv['debtor_concentration_pct']
).clip(0.35, 0.88)

recv['borrowing_base_downturn'] = recv['eligible_receivables_balance_downturn'] * recv['advance_rate_downturn']
recv['headroom_downturn'] = (recv['borrowing_base_downturn'] - recv['drawn_balance']).clip(lower=0.0)

recv['headroom_utilisation_stress'] = (
    recv['headroom_utilisation_near_default'] + 0.15 + 0.10 * recv['watchlist_flag']
).clip(0.20, 1.00)

recv['ead_downturn'] = recv['drawn_balance'] * (1.0 + 0.05 * recv['watchlist_flag'] + 0.03 * recv['dilution_pct']) + recv['headroom_downturn'] * recv['headroom_utilisation_stress']
recv['ead_downturn'] = np.maximum(recv['ead_downturn'], recv['drawn_balance'])

recv['ead_uplift_from_headroom_pct'] = (
    (recv['ead_base'] - recv['drawn_balance']) / recv['drawn_balance'].replace(0, np.nan)
).fillna(0.0).clip(0.0, 2.0)

print('=== EAD Summary ===')
ead_summary = pd.DataFrame(
    [
        {
            'loan_count': len(recv),
            'total_drawn_amount': recv['drawn_balance'].sum(),
            'total_undrawn_commitment': recv['undrawn_amount'].sum(),
            'total_ead_base': recv['ead_base'].sum(),
            'total_ead_downturn': recv['ead_downturn'].sum(),
            'mean_drawn_pct_limit': (recv['drawn_balance'] / recv['facility_limit']).mean(),
            'mean_ead_uplift_from_headroom_pct': recv['ead_uplift_from_headroom_pct'].mean(),
        }
    ]
)
display(ead_summary.round(4))


In [ ]:
# LGD logic: pool quality + controls + timing
industry_risk_factor = recv['industry_risk_bucket'].map({'Low': 0.0, 'Medium': 0.5, 'Elevated': 1.0}).fillna(0.5)

recv['lgd_base_economic'] = (
    recv['realised_lgd']
    + 0.10 * recv['debtor_concentration_pct']
    + 0.12 * recv['ageing_over_90d_pct']
    + 0.14 * recv['dilution_pct']
    + 0.10 * (1.0 - recv['collections_control_score'])
    + 0.06 * ((recv['time_to_collect_days'] - 60.0) / 140.0).clip(0.0, 1.0)
    + 0.03 * industry_risk_factor
    - 0.03 * recv['lockbox_sweep_flag']
).clip(0.03, 0.99)

# Downturn effects: slower collections, higher disputes/dilution, lower eligibility
recv['time_to_collect_days_downturn'] = (
    recv['time_to_collect_days'] * (1.18 + 0.10 * recv['watchlist_flag'] + 0.08 * recv['dilution_pct'])
).clip(25.0, 300.0)

recv['downturn_scalar'] = (
    1.00
    + 0.20 * recv['ageing_over_90d_pct']
    + 0.24 * recv['dilution_pct']
    + 0.12 * (1.0 - recv['collections_control_score'])
    + 0.10 * ((recv['time_to_collect_days_downturn'] - recv['time_to_collect_days']) / 180.0).clip(0.0, 1.0)
).clip(1.00, 1.90)

recv['lgd_downturn'] = (recv['lgd_base_economic'] * recv['downturn_scalar']).clip(0.0, 1.0)
recv['moc_addon'] = 0.025 + 0.010 * recv['watchlist_flag']
recv['lgd_final_regulatory'] = np.maximum(recv['lgd_downturn'] + recv['moc_addon'], 0.35).clip(0.0, 1.0)


def weighted_lgd(df, group_col):
    rows = []
    for k, g in df.groupby(group_col, observed=True):
        rows.append(
            {
                'dimension': group_col,
                'bucket': str(k),
                'loan_count': len(g),
                'total_ead_base': g['ead_base'].sum(),
                'ead_weighted_lgd_base': exposure_weighted_average(g, 'lgd_base_economic', 'ead_base'),
                'ead_weighted_lgd_downturn': exposure_weighted_average(g, 'lgd_downturn', 'ead_base'),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead_base'),
            }
        )
    return pd.DataFrame(rows)

weighted_by_concentration = weighted_lgd(recv, 'concentration_band')
weighted_by_ageing = weighted_lgd(recv, 'ageing_band')
weighted_by_advance = weighted_lgd(recv, 'advance_rate_band')

segment_summary = pd.concat(
    [weighted_by_concentration, weighted_by_ageing, weighted_by_advance],
    ignore_index=True,
)

base_vs_downturn = pd.DataFrame(
    [
        {
            'ead_weighted_lgd_base': exposure_weighted_average(recv, 'lgd_base_economic', 'ead_base'),
            'ead_weighted_lgd_downturn': exposure_weighted_average(recv, 'lgd_downturn', 'ead_base'),
            'ead_weighted_lgd_final': exposure_weighted_average(recv, 'lgd_final_regulatory', 'ead_base'),
        }
    ]
)
base_vs_downturn['downturn_sensitivity_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_downturn'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)
base_vs_downturn['final_minus_base_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_final'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)

print('=== Weighted LGD by Concentration Band ===')
display(weighted_by_concentration.round(4))
print('=== Weighted LGD by Ageing Band ===')
display(weighted_by_ageing.round(4))
print('=== Weighted LGD by Advance-Rate Band ===')
display(weighted_by_advance.round(4))
print('=== Base vs Downturn ===')
display(base_vs_downturn.round(4))


In [ ]:
# Sensitivity analysis

def run_sensitivity(df, ageing_add=0.0, dilution_add=0.0, ineligible_add=0.0, collect_days_mult=1.0):
    x = df.copy()
    x['ageing_s'] = (x['ageing_over_90d_pct'] + ageing_add).clip(0.0, 0.70)
    x['dilution_s'] = (x['dilution_pct'] + dilution_add).clip(0.0, 0.45)
    x['ineligible_s'] = (x['ineligible_receivables_pct'] + ineligible_add).clip(0.0, 0.80)
    x['collect_days_s'] = (x['time_to_collect_days'] * collect_days_mult).clip(20.0, 360.0)

    lgd_base_s = (
        x['realised_lgd']
        + 0.10 * x['debtor_concentration_pct']
        + 0.12 * x['ageing_s']
        + 0.14 * x['dilution_s']
        + 0.10 * (1.0 - x['collections_control_score'])
        + 0.06 * ((x['collect_days_s'] - 60.0) / 140.0).clip(0.0, 1.0)
        + 0.03 * x['industry_risk_bucket'].map({'Low': 0.0, 'Medium': 0.5, 'Elevated': 1.0}).fillna(0.5)
        - 0.03 * x['lockbox_sweep_flag']
    ).clip(0.03, 0.99)

    downturn_scalar_s = (
        1.00
        + 0.20 * x['ageing_s']
        + 0.24 * x['dilution_s']
        + 0.12 * (1.0 - x['collections_control_score'])
        + 0.10 * ((x['collect_days_s'] - x['time_to_collect_days']) / 180.0).clip(0.0, 1.0)
        + 0.06 * x['ineligible_s']
    ).clip(1.00, 2.00)

    lgd_down_s = (lgd_base_s * downturn_scalar_s).clip(0.0, 1.0)
    lgd_final_s = np.maximum(lgd_down_s + 0.025 + 0.01 * x['watchlist_flag'], 0.35).clip(0.0, 1.0)

    return {
        'ead_weighted_lgd_base': exposure_weighted_average(x.assign(lgd_base_s=lgd_base_s), 'lgd_base_s', 'ead_base'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(x.assign(lgd_down_s=lgd_down_s), 'lgd_down_s', 'ead_base'),
        'ead_weighted_lgd_final': exposure_weighted_average(x.assign(lgd_final_s=lgd_final_s), 'lgd_final_s', 'ead_base'),
    }

scenarios = [
    {'scenario': 'base', 'ageing_add': 0.00, 'dilution_add': 0.00, 'ineligible_add': 0.00, 'collect_days_mult': 1.00},
    {'scenario': 'ageing_+5pp', 'ageing_add': 0.05, 'dilution_add': 0.00, 'ineligible_add': 0.00, 'collect_days_mult': 1.00},
    {'scenario': 'dilution_+3pp', 'ageing_add': 0.00, 'dilution_add': 0.03, 'ineligible_add': 0.00, 'collect_days_mult': 1.00},
    {'scenario': 'eligibility_-5pp', 'ageing_add': 0.00, 'dilution_add': 0.00, 'ineligible_add': 0.05, 'collect_days_mult': 1.00},
    {'scenario': 'slow_collections_20pct', 'ageing_add': 0.00, 'dilution_add': 0.00, 'ineligible_add': 0.00, 'collect_days_mult': 1.20},
    {'scenario': 'combined_stress', 'ageing_add': 0.05, 'dilution_add': 0.03, 'ineligible_add': 0.05, 'collect_days_mult': 1.20},
]

sens_rows = []
for s in scenarios:
    m = run_sensitivity(
        recv,
        ageing_add=s['ageing_add'],
        dilution_add=s['dilution_add'],
        ineligible_add=s['ineligible_add'],
        collect_days_mult=s['collect_days_mult'],
    )
    sens_rows.append({'scenario': s['scenario'], **m})

sensitivity = pd.DataFrame(sens_rows)
print('=== Sensitivity Analysis ===')
display(sensitivity.round(4))


In [ ]:
# Monitoring, validation, charts, exports
recv['default_year'] = pd.to_datetime(recv['default_date']).dt.year
monitoring = (
    recv.groupby('default_year', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead_base': g['ead_base'].sum(),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead_base'),
                'mean_time_to_collect_days': g['time_to_collect_days'].mean(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

validation_checks = pd.DataFrame(
    [
        {'check_name': 'eligible_balance_not_above_gross', 'passed': (recv['eligible_receivables_balance'] <= recv['gross_receivables_balance']).all()},
        {'check_name': 'ineligible_pct_between_0_and_1', 'passed': recv['ineligible_receivables_pct'].between(0, 1).all()},
        {'check_name': 'ead_base_not_below_drawn', 'passed': (recv['ead_base'] >= recv['drawn_balance']).all()},
        {'check_name': 'ead_downturn_not_below_drawn', 'passed': (recv['ead_downturn'] >= recv['drawn_balance']).all()},
        {'check_name': 'time_to_collect_positive', 'passed': (recv['time_to_collect_days'] > 0).all()},
        {'check_name': 'downturn_collection_time_longer', 'passed': (recv['time_to_collect_days_downturn'] >= recv['time_to_collect_days']).all()},
    ]
)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = segment_summary[segment_summary['dimension'] == 'concentration_band']
ax.bar(plot_df['bucket'], plot_df['ead_weighted_lgd_final'], color='#c44e52')
ax.set_title('Receivables LGD: Weighted Final LGD by Concentration Band')
ax.set_ylabel('EAD-weighted Final LGD')
ax.set_xlabel('Concentration Band')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'receivables_invoice_finance_lgd_by_concentration.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(sensitivity['scenario'], sensitivity['ead_weighted_lgd_downturn'], color='#dd8452')
ax.set_title('Receivables LGD Sensitivity: EAD-weighted Downturn LGD')
ax.set_ylabel('Downturn LGD')
ax.set_xlabel('Scenario')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'receivables_invoice_finance_sensitivity.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

loan_level_output = recv[
    [
        'loan_id', 'facility_type', 'security_type', 'industry', 'industry_risk_bucket',
        'drawn_balance', 'undrawn_amount', 'ead', 'ead_base', 'ead_downturn',
        'gross_receivables_balance', 'eligible_receivables_balance', 'ineligible_receivables_pct',
        'debtor_concentration_pct', 'ageing_over_90d_pct', 'dilution_pct',
        'collections_control_score', 'lockbox_sweep_flag', 'advance_rate',
        'time_to_collect_days', 'time_to_collect_days_downturn',
        'concentration_band', 'ageing_band', 'advance_rate_band',
        'realised_lgd', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory',
    ]
].copy()

loan_level_output.to_csv(os.path.join(TABLE_DIR, 'receivables_invoice_finance_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(TABLE_DIR, 'receivables_invoice_finance_segment_summary.csv'), index=False)
ead_summary.to_csv(os.path.join(TABLE_DIR, 'receivables_invoice_finance_ead_summary.csv'), index=False)
base_vs_downturn.to_csv(os.path.join(TABLE_DIR, 'receivables_invoice_finance_base_vs_downturn.csv'), index=False)
sensitivity.to_csv(os.path.join(TABLE_DIR, 'receivables_invoice_finance_sensitivity.csv'), index=False)
monitoring.to_csv(os.path.join(TABLE_DIR, 'receivables_invoice_finance_monitoring_by_year.csv'), index=False)
validation_checks.to_csv(os.path.join(TABLE_DIR, 'receivables_invoice_finance_validation_checks.csv'), index=False)

print('=== Validation Checks ===')
display(validation_checks)

print('Saved receivables outputs:')
print('- receivables_invoice_finance_loan_level_output.csv')
print('- receivables_invoice_finance_segment_summary.csv')
print('- receivables_invoice_finance_ead_summary.csv')
print('- receivables_invoice_finance_base_vs_downturn.csv')
print('- receivables_invoice_finance_sensitivity.csv')
print('- receivables_invoice_finance_monitoring_by_year.csv')
print('- receivables_invoice_finance_validation_checks.csv')
